In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def _find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "experiments" / "real_data").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks.helpers import real_transport as rt

RESULTS_ROOT = REPO_ROOT / "results" / "real_data"
print(f"Results root: {RESULTS_ROOT.relative_to(REPO_ROOT)}")


## 1. Runs on disk

`find_runs` lists every transport evaluation under `results/real_data/`. One row per
(dataset, held-out window, seed) cell.


In [ ]:
runs = rt.find_runs(RESULTS_ROOT)
table = [(rt.DATASET_LABELS.get(d, d), w, s) for d, w, s, _ in runs]
pd.DataFrame(table, columns=["dataset", "held-out window", "seed"])


In [ ]:
summary = rt.summarize(rt.collect(RESULTS_ROOT))


def fmt(entry, arm):
    return f"{entry[arm][0]:.2f} ± {entry[arm][1]:.2f}"


rows = []
for dataset, entry in summary.items():
    ours = entry["linear_projected"][0]
    ot_cfm = entry["ot_cfm"][0]
    gaga = rt.GAGA_PUBLISHED.get(dataset)
    rows.append(
        {
            "dataset": rt.DATASET_LABELS.get(dataset, dataset),
            "ours (projector)": fmt(entry, "linear_projected"),
            "chord refined": fmt(entry, "chord_refined_pairs"),
            "OT-CFM": fmt(entry, "ot_cfm"),
            "I-CFM": fmt(entry, "i_cfm"),
            "GAGA (publ.)": f"{gaga:.2f}" if gaga else "—",
            "vs OT-CFM": f"{100 * (ours - ot_cfm) / ot_cfm:+.1f}%",
        }
    )
pd.DataFrame(rows).set_index("dataset")


### Do the gaps clear their confidence intervals?

With three seeds the 95 % CI is $t(0.975, 2)\,s/\sqrt{3}$. A margin only counts if the
intervals are disjoint.


In [ ]:
for dataset, entry in summary.items():
    label = rt.DATASET_LABELS.get(dataset, dataset)
    mean, half = entry["linear_projected"][0], entry["_ci95"]["linear_projected"]
    for baseline in ("ot_cfm", "i_cfm", "chord_refined_pairs"):
        mean_b, half_b = entry[baseline][0], entry["_ci95"][baseline]
        verdict = "disjoint" if mean + half < mean_b - half_b else "OVERLAP"
        print(
            f"  {label:10s} vs {baseline:19s}: "
            f"[{mean - half:6.2f},{mean + half:6.2f}] vs "
            f"[{mean_b - half_b:6.2f},{mean_b + half_b:6.2f}]   {verdict}"
        )


## 3. Which part of the geometry is responsible?

Each construction is scored before and after passing through the projector.


In [ ]:
PAIRS = [
    ("interpolant", "linear"),
    ("OT-CFM", "ot_cfm"),
    ("I-CFM", "i_cfm"),
    ("chord refined", "chord_refined_pairs"),
]

rows = []
for dataset, entry in summary.items():
    row = {"dataset": rt.DATASET_LABELS.get(dataset, dataset)}
    for label, arm in PAIRS:
        row[f"{label} raw"] = entry[arm][0]
        row[f"{label} +P"] = entry[f"{arm}_projected"][0]
    rows.append(row)

df = pd.DataFrame(rows).set_index("dataset").round(2)
display(df)

projected = [column for column in df.columns if "+P" in column]
print("\nspread across constructions after projection, vs the 95% CI:")
for dataset, entry in summary.items():
    label = rt.DATASET_LABELS.get(dataset, dataset)
    spread = df.loc[label, projected].max() - df.loc[label, projected].min()
    ci = max(entry["_ci95"][f"{arm}_projected"] for _, arm in PAIRS)
    verdict = "indistinguishable" if spread < 2 * ci else "DISTINGUISHABLE"
    print(f"  {label:10s} spread {spread:.2f}   CI ±{ci:.2f}   {verdict}")
